# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps in identifying the main data tables and how to refer to fields using their `@id` values.

In [ ]:
# Print all record sets and their @id
print('Record Sets in the dataset:')
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")

# For each record set, print available fields and columns (@id)
for rs in record_sets:
    print(f"\nFields in Record Set {rs.id}:")
    for field in rs.fields:
        print(f"  - Field @id: {field.id}, name: {getattr(field, 'name', '')}, data_type: {getattr(field, 'data_type', '')}")
    print('Columns:')
    for col in rs.columns:
        print(f"  - Column @id: {col.id}, name: {getattr(col, 'name', '')}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis, referencing entities using their `@id`. Here, we extract the records for each record set found in the previous step.

In [ ]:
# Extract data from each record set
all_dataframes = {}

# Build a list of record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print('Record set @ids:', record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    all_dataframes[record_set_id] = pd.DataFrame(records)
    print(f'Loaded {len(records)} records for Record Set {record_set_id}\n')

# Let's preview the columns for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f'Columns in DataFrame for Record Set {first_rs}:')
    print(all_dataframes[first_rs].columns.tolist())
    display(all_dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply exploratory and preprocessing steps, always referencing fields by their `@id`.

In [ ]:
# Let's select the first record set and a numeric field for demonstration

rs_id = record_set_ids[0]
df = all_dataframes[rs_id]

# Get the list of numeric fields from the record set
numeric_fields = []
for field in dataset.metadata.record_set(rs_id).fields:
    if hasattr(field, 'data_type') and field.data_type in ['schema:Float', 'schema:Number', 'schema:Integer']:
        numeric_fields.append(field.id)

print(f'Numeric fields under Record Set {rs_id} (by @id):')
print(numeric_fields)

# Pick the first available numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # by convention, this is the @id
    print('Analyzing numeric field:', numeric_field_id)
    # To avoid missing columns, check if @id is in DataFrame
    if numeric_field_id in df.columns:
        # Make sure column is numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Normalize the field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a non-numeric field (e.g. the first String/Text field)
        group_field = None
        for field in dataset.metadata.record_set(rs_id).fields:
            if hasattr(field, 'data_type') and field.data_type in ['schema:Text']:
                if field.id in df.columns:
                    group_field = field.id
                    break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field}':\n", grouped_df.head())
        else:
            print('No text/group field found for grouping.')
    else:
        print(f"Column {numeric_field_id} not present in DataFrame for Record Set {rs_id}.")
else:
    print(f'No numeric fields found in Record Set {rs_id}.')

## 5. Visualization
Visualize distributions or relationships between fields using the referenced `@id` columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# We will show distribution of the numeric field and boxplot for groups (if available)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f'Distribution of Field @{numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.show()
        # Try boxplot by group
        if 'group_field' in locals() and group_field is not None:
            plt.figure(figsize=(12,6))
            sns.boxplot(data=df, x=group_field, y=numeric_field_id)
            plt.title(f'Boxplot of {numeric_field_id} by {group_field}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the Croissant dataset and examined metadata using the `mlcroissant` library,
- Explored available record sets, fields, and their unique `@id`s,
- Loaded and previewed the records in tabular DataFrames,
- Performed basic EDA by filtering, normalizing, and grouping data (referencing columns by `@id`),
- Visualized numeric field distributions and group statistics.

This workflow demonstrates a transparent and reproducible approach to FAIR data processing using Croissant schemas. For advanced analyses, extend these explorations to additional record sets, fields, and custom processing as needed.